In [15]:
import json
import os
import pandas as pd

train_path = "D:\.Projects\\Topic202507Codes\\data\\redial_dataset\\train_data.jsonl"
with open(train_path, "r", encoding="utf-8") as f:
    lines = f.readlines()
train_data = [json.loads(line) for line in lines]

test_path = "D:\.Projects\\Topic202507Codes\\data\\redial_dataset\\test_data.jsonl"
with open(train_path, "r", encoding="utf-8") as f:
    lines = f.readlines()
test_data = [json.loads(line) for line in lines]

movie_id2title = {}
movie_title2id = {}
movie_path = "D:\.Projects\\Topic202507Codes\\data\\redial_dataset\\movies_with_mentions.csv"
movie_mentions = pd.read_csv(movie_path)
for _, row in movie_mentions.iterrows():
    id = row["movieId"]
    title = row["movieName"]
    movie_id2title[id] = title
    movie_title2id[title] = id

In [16]:
import nltk

nltk.download("punkt")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Administrator\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Administrator\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [23]:
from nltk.tokenize import word_tokenize, RegexpTokenizer
from nltk.corpus import stopwords
from tqdm import tqdm
import re

# def clean_text(text):\


def process(data, movie_patter=r"@\d+"):
    processed_data = []
    tokenizer = RegexpTokenizer(r"@\d+|[\w\']+|[^\w\s]")
    stop_words = set(stopwords.words("english"))
    for conv in tqdm(data):
        conv_dict = {}
        conv_seeker = conv["initiatorWorkerId"]
        conv_dict["conv_id"] = conv["conversationId"]
        conv_dict["dialog"] = []
        for utt_id, utt in enumerate(conv["messages"]):
            utt_dict = {}
            utt_dict["utt_id"] = utt_id
            utt_dict["role"] = "Seeker" if utt["senderWorkerId"] == conv_seeker else "Recommender"

            movies = re.findall(movie_patter, utt["text"])
            movies_ids = [int(mv[1:]) for mv in movies if int(mv[1:]) in movie_id2title]
            utt_dict["movies"] = movies_ids

            tokens = tokenizer.tokenize(utt["text"])
            filtered_tokens = [word.lower() for word in tokens if word.lower() not in stop_words]
            utt_dict["text"] = filtered_tokens

            conv_dict["dialog"].append(utt_dict)
        processed_data.append(conv_dict)
    return processed_data

In [24]:
processed_train = process(train_data)
processed_test = process(test_data)

100%|██████████| 10006/10006 [00:02<00:00, 4047.13it/s]


In [25]:
print(json.dumps(processed_train[0], indent=2))

{
  "conv_id": "391",
  "dialog": [
    {
      "utt_id": 0,
      "role": "Seeker",
      "movies": [],
      "text": [
        "hi",
        ",",
        "?",
        "looking",
        "movie",
        "recommendations"
      ]
    },
    {
      "utt_id": 1,
      "role": "Recommender",
      "movies": [],
      "text": [
        "okay",
        ".",
        "kind",
        "movies",
        "like",
        "?"
      ]
    },
    {
      "utt_id": 2,
      "role": "Seeker",
      "movies": [
        84779,
        191602
      ],
      "text": [
        "like",
        "animations",
        "like",
        "@84779",
        "@191602"
      ]
    },
    {
      "utt_id": 3,
      "role": "Seeker",
      "movies": [
        122159
      ],
      "text": [
        "also",
        "enjoy",
        "@122159"
      ]
    },
    {
      "utt_id": 4,
      "role": "Seeker",
      "movies": [],
      "text": [
        "anything",
        "artistic"
      ]
    },
    {
      "utt_id": 5,
  

In [29]:
output_dir = os.path.join(os.getcwd(), "redial")
os.makedirs(output_dir, exist_ok=True)
with open(os.path.join(output_dir, "train_data.json"), "w", encoding="utf-8") as f:
    json.dump(processed_train, f, indent=2)
with open(os.path.join(output_dir, "test_data.json"), "w", encoding="utf-8") as f:
    json.dump(processed_test, f, indent=2)

In [30]:
token2id = {"__pad__": 0, "__start__": 1, "__end__": 2, "__unk__": 3}
for conv in processed_train + processed_test:
    for utt in conv["dialog"]:
        for token in utt["text"]:
            if token not in token2id:
                token2id[token] = len(token2id)
with open(os.path.join(output_dir, "token2id.json"), "w", encoding="utf-8") as f:
    json.dump(token2id, f, indent=2)